# NB06 — Luki-style SAM1 recovery | Baego
**Goal:** visually inspect the original successful Block 53 SAM1 ViT-H predictions and NMS alternative, already saved for Baego ROI 1 and ROI 4. No GPU, inference, training, or destructive writes.
The Luki site remains part of the project but is **not displayed until its own verified saved predictions are identified**. Reported F1 values come from archived CSV, not fresh evaluation.

In [ ]:
# 01 — Setup (read-only paths)
from pathlib import Path
from google.colab import drive
import json, re, warnings
import numpy as np, pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from shapely.geometry import box
from rasterio.windows import from_bounds
from rasterio.windows import transform as window_transform

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Musanga')
DATA = ROOT / 'data/Geo_Ref'
ARCHIVE = ROOT / 'KVO_work/KVO_06_UNet_guided_SAM3'
ARCHIVE_V2 = ROOT / 'KVO_work/KVO_06_v2_Baego_Luki'
INPUTS = {'Baego_RGB': DATA/'Baego_georef.tif', 'Baego_Kristof': DATA/'Annotations/Crown_Kristof.shp', 'Baego_Nadine': DATA/'Annotations/Crown_Nadine.shp', 'Baego_ROI': DATA/'Annotations/ROI_Crowns_Kvo.shp', 'Luki_RGB': DATA/'Luki.tif', 'Luki_reference': DATA/'Annotations/Tree_Annotation_Luki.gpkg'}
for name,p in INPUTS.items(): print(f'{name:18} {"OK" if p.exists() else "MISSING"}  {p}')
print('Historical folders:', [(str(p),p.exists()) for p in (ARCHIVE,ARCHIVE_V2)])

In [ ]:
# 02 — Reference crowns and area rule (metres squared)
MIN_CROWN_M2 = 5.0

def load_layer(path, target_crs, raster_bounds, layer=None):
    g = gpd.read_file(path, layer=layer) if layer else gpd.read_file(path)
    g = g.loc[g.geometry.notna() & ~g.geometry.is_empty].copy()
    if g.empty:
        print(f'WARNING: no geometries in {path.name}')
        return g

    # A missing .prj is common in older annotation shapefiles.
    # Assign raster CRS only when the raw coordinate extents already overlap.
    if g.crs is None:
        xmin, ymin, xmax, ymax = g.total_bounds
        rxmin, rymin, rxmax, rymax = tuple(raster_bounds)
        overlaps = xmax >= rxmin and xmin <= rxmax and ymax >= rymin and ymin <= rymax
        if not overlaps:
            raise ValueError(
                f'{path.name} has no CRS and its coordinates do not overlap the RGB raster. '
                f'Annotation bounds={g.total_bounds}; raster bounds={tuple(raster_bounds)}. '
                'Check the original annotation CRS before proceeding.'
            )
        print(f'WARNING: {path.name} has no CRS; coordinates overlap raster. '
              f'Assuming raster CRS {target_crs}.')
        g = g.set_crs(target_crs, allow_override=True)
    else:
        g = g.to_crs(target_crs)
    return g

def load_crowns(path, target_crs, raster_bounds, layer=None):
    g = load_layer(path, target_crs, raster_bounds, layer)
    if not g.crs.is_projected:
        raise ValueError('A projected metric CRS is required for the 5 m² crown filter')
    if g.crs.axis_info and not all(
        axis.unit_name.lower() in ('metre','meter','metres','meters') for axis in g.crs.axis_info[:2]
    ):
        raise ValueError(f'CRS units are not metres: {g.crs}')
    before = len(g)
    g = g.loc[g.geometry.area >= MIN_CROWN_M2].copy()
    print(f'  {path.name}: {before} polygons → {len(g)} crowns ≥ {MIN_CROWN_M2:g} m²')
    return g

SITES = {}
for site, rgb_key in [('Baego','Baego_RGB'),('Luki','Luki_RGB')]:
    if not INPUTS[rgb_key].exists(): continue
    with rasterio.open(INPUTS[rgb_key]) as ds:
        if ds.crs is None or not ds.crs.is_projected:
            raise ValueError(f'{site}: projected raster CRS required')
        site_info = {'rgb':INPUTS[rgb_key], 'crs':ds.crs,
                     'bounds':ds.bounds, 'res':ds.res}
    crs, bounds = site_info['crs'], site_info['bounds']
    print(f'\n{site}: raster CRS={crs}, resolution={site_info["res"]}')
    if site == 'Baego':
        site_info['Kristof'] = load_crowns(INPUTS['Baego_Kristof'], crs, bounds)
        site_info['Nadine'] = load_crowns(INPUTS['Baego_Nadine'], crs, bounds)
        site_info['roi'] = load_layer(INPUTS['Baego_ROI'], crs, bounds)
    else:
        site_info['reference'] = load_crowns(
            INPUTS['Luki_reference'], crs, bounds,
            layer='photointerpretation_inventory')
    SITES[site] = site_info
    print('Reference counts:', {
        k:len(v) for k,v in site_info.items()
        if isinstance(v,gpd.GeoDataFrame)
    })

In [ ]:
# 03 — Frozen Luki-inspired Block 53 settings (archived provenance)
# No model initialization or inference. These are the settings recovered from prior experiment notes.
LUKI_STYLE = {
    'model': 'SAM1 ViT-H, automatic mask generation',
    'points_per_side': 32,
    'pred_iou_thresh': 0.88,
    'stability_score_thresh': 0.95,
    'core_size_m': 40,
    'context_buffer_each_side_m': 60,
    'context_extent_m': 160,
    'selection': 'mask centroid inside 40 m core',
    'components': 'up to three largest per mask',
    'predicted_area_range_m2': (2, 2500),
    'postprocessing': ['author_style', 'nms_060'],
}
print('Historical Block 53 method (recovered notes):')
for key,value in LUKI_STYLE.items(): print(f'  {key}: {value}')
print('\nSource: archived experiment v53. These are not newly inferred SAM settings.')
print('Reference crown cutoff remains 5 m²; historical predictions are NOT re-filtered.')

In [ ]:
# 04 — Load ONLY the saved Block 53 Baego predictions and comparison metrics
# This exact archive has ROI 1 and ROI 4. No historical outputs are modified.
HISTORY = ARCHIVE / 'experiments_v53_luki_sam_only_replication'
if not HISTORY.is_dir(): raise FileNotFoundError(f'Block 53 archive not found: {HISTORY}')
metric_path = HISTORY / 'luki_style_roi_comparison.csv'
if not metric_path.exists(): raise FileNotFoundError(metric_path)
METRICS = pd.read_csv(metric_path)
print('=== ORIGINAL BLOCK 53: SAVED F1 AT IoU >= 0.5 ===')
display(METRICS.loc[METRICS['iou_threshold'].eq(0.5),
    ['roi','postprocess','reference','predicted','matched','precision','recall','f1']
].sort_values(['roi','postprocess']).reset_index(drop=True).round(3))

PREDICTIONS_BY_ROI = {}
for roi_id in (1,4):
    for method in ('author_style','nms_060'):
        p=HISTORY / f'roi{roi_id}_{method}.gpkg'
        if not p.exists(): raise FileNotFoundError(p)
        g=gpd.read_file(p)
        if g.crs is None:
            raise ValueError(f'Saved predictions missing CRS: {p.name}; do not guess.')
        g=g.to_crs(SITES['Baego']['crs'])
        PREDICTIONS_BY_ROI[(roi_id,method)] = g
        print(f'ROI {roi_id:>1}, {method:12}: {len(g)} saved predictions ({p.name})')

# Check that predictions really overlap the numbered ROI; fail instead of showing unrelated geometry.
baego_rois=SITES['Baego']['roi']
for roi_id in (1,4):
    if roi_id>=len(baego_rois): raise ValueError(f'Missing Baego ROI index {roi_id}')
    roi_geom=baego_rois.geometry.iloc[roi_id]
    for method in ('author_style','nms_060'):
        g=PREDICTIONS_BY_ROI[(roi_id,method)]
        in_roi=g.geometry.intersects(roi_geom).sum()
        print(f'ROI {roi_id} {method}: {in_roi}/{len(g)} saved polygons intersect the current ROI geometry')
        if in_roi==0:
            raise ValueError('ROI order likely changed. Resolve mapping before visualization.')
print('\nReady: Block 05 shows the original author-style and NMS crown boundaries.')

In [ ]:
# 05 — Visual comparison: original Block 53 vs NMS on RGB
# Simply change VIEW_ROI to 1 or 4. No inference, no writes.
VIEW_ROI = 1  # 1 or 4
if VIEW_ROI not in (1,4): raise ValueError('Block 53 has saved outputs only for ROI 1 and ROI 4.')
s=SITES['Baego']
roi_geom=s['roi'].geometry.iloc[VIEW_ROI]
bounds=roi_geom.bounds
with rasterio.open(s['rgb']) as ds:
    from rasterio.windows import Window
    w=from_bounds(*bounds,transform=ds.transform).round_offsets().round_lengths()
    w=w.intersection(Window(0,0,ds.width,ds.height))
    if w.width<=0 or w.height<=0: raise ValueError('ROI lies outside Baego RGB.')
    rgb=ds.read([1,2,3],window=w,
                out_shape=(3,min(int(w.height),1600),min(int(w.width),1600)))
    bb=rasterio.windows.bounds(w,ds.transform)
img=np.moveaxis(rgb,0,-1).astype(float)
lo,hi=np.nanpercentile(img,[2,98])
img=np.clip((img-lo)/(hi-lo+1e-9),0,1)
extent=[bb[0],bb[2],bb[1],bb[3]]
area=box(*bb)
fig,axes=plt.subplots(1,2,figsize=(16,8),constrained_layout=True)
for ax,method,title in zip(axes,('author_style','nms_060'),
                            ('Luki-style SAM1 ViT-H','Luki-style SAM1 + NMS')):
    ax.imshow(img,extent=extent)
    for label,color in (('Kristof','white'),('Nadine','yellow')):
        g=s[label]
        g=g.loc[g.geometry.intersects(area)]
        if len(g): g.boundary.plot(ax=ax,color=color,linewidth=0.8)
    g=PREDICTIONS_BY_ROI[(VIEW_ROI,method)]
    visible=g.loc[g.geometry.intersects(area)]
    if len(visible): visible.boundary.plot(ax=ax,color='red',linewidth=1)
    score=METRICS.loc[(METRICS.roi==VIEW_ROI)&(METRICS.postprocess==method)&
                       (METRICS.iou_threshold==0.5),'f1']
    score_text=f'{float(score.iloc[0]):.3f}' if len(score) else 'n/a'
    ax.set_title(f'{title}\nF1={score_text} | {len(g)} saved crowns')
    ax.set_xlim(bb[0],bb[2]);ax.set_ylim(bb[1],bb[3])
    ax.set_aspect('equal')
fig.suptitle(f'Baego ROI {VIEW_ROI} | white=Kristof, yellow=Nadine, red=SAM')
plt.show()
print('Next: switch VIEW_ROI between 1 and 4. Judge complete individual crowns and seam artifacts.')

In [ ]:
# 06 — Brief assessment summary (original metrics, NOT recalculated)
d=METRICS.loc[METRICS.iou_threshold==0.5]
print('Historical Block 53, IoU >= 0.5')
display(d.pivot(index='roi',columns='postprocess',values='f1').round(3))
print('ROI 1 and 4 only. Do not generalize these scores to Luki or all six Baego ROIs.')
print('Nadine comparisons are a separate sparse-reference benchmark; not pooled with Kristof F1.')
print('Luki is not run in this notebook: the historical Luki-site result set remains to be identified.')
print('No new SAM inference or training; no original outputs modified.')

## 07 — Decision log
- **Recovered method:** SAM1 ViT-H, automatic mask generation, 40 m cores + 60 m context on each side (Block 53).
- **Archived inputs:** `experiments_v53_luki_sam_only_replication/roi{1,4}_{author_style,nms_060}.gpkg`
- **Expected F1 at IoU 0.5:** ROI 1 author-style 0.493 / NMS 0.517; ROI 4 author-style 0.394 / NMS 0.417.
- **Next decision:** visually inspect ROI 1 and ROI 4 (Block 05) before deciding whether any new inference is justified.
- **Pending:** locate and verify original Luki-site saved predictions and exact historical six-ROI baseline.
